## Įterpimai

Mūsų ankstesniame pavyzdyje dirbome su aukštos matmenų žodžių maišo vektoriais, kurių ilgis buvo `vocab_size`, ir mes aiškiai konvertavome iš žemos matmenų pozicijos reprezentacijos vektorių į retą vienkartinę reprezentaciją. Ši vienkartinė reprezentacija nėra atminties efektyvi, be to, kiekvienas žodis yra traktuojamas nepriklausomai nuo kitų, t. y. vienkartiniu kodu pažymėti vektoriai nereikš jokios semantinės panašumo tarp žodžių.

Šiame skyriuje mes tęsiame **News AG** duomenų rinkinio tyrimą. Pradžiai, įkelkime duomenis ir gaukime keletą apibrėžimų iš ankstesnio užrašų knygelės.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Kas yra embedding?

Idėja **embedding** yra žodžius atvaizduoti mažesnės dimensijos tankiais vektoriais, kurie tam tikru būdu atspindi žodžio semantinę prasmę. Vėliau aptarsime, kaip sukurti prasmingus žodžių embeddingus, tačiau dabar galvokime apie embeddingus kaip apie būdą sumažinti žodžio vektoriaus dimensiją.

Taigi, embedding sluoksnis priims žodį kaip įvestį ir sukurs išeinantį vektorių nurodytos `embedding_size` dimensijos. Tam tikra prasme tai labai panašu į `Linear` sluoksnį, bet vietoje vieno karšto koduotės vektoriaus jis galės priimti žodžio numerį kaip įvestį.

Naudodami embedding sluoksnį kaip pirmąjį mūsų tinklo sluoksnį, galime pereiti nuo žodžių maišo modelio prie **embedding maišo** modelio, kuriame pirmiausia kiekvieną teksto žodį paverčiame atitinkamu embeddingu ir tada apskaičiuojame tam tikrą funkciją, naudojant visus tuos embeddingus, pavyzdžiui, `sum`, `average` arba `max`.

![Paveikslėlis, vaizduojantis embedding klasifikatorių penkiems sekos žodžiams.](../../../../../translated_images/lt/embedding-classifier-example.b77f021a7ee67eee.webp)

Mūsų klasifikatoriaus neuroninis tinklas prasidės embedding sluoksniu, po to suvestinių sluoksniu ir galiausiai linijiniu klasifikatoriumi viršuje:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Kintamo sekos dydžio tvarkymas

Dėl šios architektūros mums reikės tam tikru būdu kurti minibatch'us mūsų tinklui. Ankstesniame skyriuje, naudojant bag-of-words, visi BoW tenesorai minibatche turėjo vienodą dydį `vocab_size`, nepriklausomai nuo tikros mūsų teksto sekos ilgio. Kai pereisime prie žodžių įterpimų, kiekviename teksto pavyzdyje turėsime skirtingą žodžių skaičių, ir sujungdami tuos pavyzdžius į minibatch'us turėsime naudoti užpildymą.

Tai galima padaryti panaudojant tą pačią techniką, pateikiant `collate_fn` funkciją duomenų šaltiniui:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Įterpimo klasifikatoriaus treniravimas

Dabar, kai mes apibrėžėme tinkamą duomenų įkroviklį, galime treniruoti modelį naudodami treniravimo funkciją, kurią apibrėžėme ankstesniame skyriuje:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Pastaba**: Čia mes treniruojame tik 25 tūkstančius įrašų (mažiau nei vieną pilną epochą) dėl laiko apribojimo, tačiau galite tęsti treniruotę, parašyti funkciją keliose epochose treniruotis ir eksperimentuoti su mokymosi spartos parametru, kad pasiektumėte didesnį tikslumą. Turėtumėte sugebėti pasiekti maždaug 90% tikslumą.


### EmbeddingBag sluoksnis ir kintamo ilgio sekų atvaizdavimas

Ankstesnėje architektūroje turėjome papildyti visas sekas iki vienodo ilgio, kad jas sutalpintume į mažą partiją. Tai nėra efektyviausias būdas atvaizduoti kintamo ilgio sekas – kitas požiūris būtų naudoti **offset** vektorių, kuris saugotų visų sekų poslinkius viename dideliame vektoriuje.

![Vaizdas, kuriame parodytas poslinkio sekų atvaizdavimas](../../../../../translated_images/lt/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Pastaba**: Aukščiau esančiame paveikslėlyje rodoma simbolių seka, tačiau mūsų pavyzdyje dirbame su žodžių sekų. Vis dėlto bendra sekų atvaizdavimo poslinkio vektoriumi principas lieka tas pats.

Norint dirbti su poslinkio atvaizdavimu, naudojame [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) sluoksnį. Jis panašus į `Embedding`, tačiau priima turinio vektorių ir poslinkio vektorių kaip įvestis, taip pat apima vidurkinimo sluoksnį, kuris gali būti `mean`, `sum` arba `max`.

Čia pateiktas modifikuotas tinklas, kuris naudoja `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Norėdami paruošti duomenų rinkinį mokymui, turime pateikti konvertavimo funkciją, kuri paruoš užlaikymo vektorių:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Atkreipkite dėmesį, kad, skirtingai nei visuose ankstesniuose pavyzdžiuose, mūsų tinklas dabar priima du parametrus: duomenų vektorių ir poslinkio vektorių, kurie yra skirtingo dydžio. Panašiai, mūsų duomenų įkėlėjas taip pat pateikia 3 reikšmes vietoj 2: tiek teksto, tiek poslinkio vektoriai pateikiami kaip ypatybės. Todėl turime šiek tiek pakoreguoti mūsų apmokymo funkciją, kad tai apsvarstytume:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantiniai įterpiniai: Word2Vec

Ankstesniame mūsų pavyzdyje modelio įterpimo sluoksnis išmoko žodžius susieti su vektorinėmis reprezentacijomis, tačiau ši reprezentacija neturėjo daug semantinės prasmės. Būtų naudinga išmokti tokią vektorinę reprezentaciją, kad panašūs žodžiai ar sinonimai atitiktų vektorius, kurie yra arti vienas kito tam tikru vektorių atstumo požiūriu (pvz., Euklido atstumu).

Tam reikia iš anksto apmokyti mūsų įterpimo modelį didelėje teksto kolekcijoje specifiniu būdu. Vienas iš pirmųjų semantinių įterpinių mokymo būdų vadinamas [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Jis pagrįstas dviem pagrindinėmis architektūromis, kurios naudojamos žodžių paskirstytajai reprezentacijai kurti:

 - **Nuolatinė žodžių krepšelio (Continuous bag-of-words, CBoW)** — šioje architektūroje mokome modelį numatyti žodį pagal jį supantį kontekstą. Duotame ngramoje $(W_{-2},W_{-1},W_0,W_1,W_2)$ modelio tikslas yra numatyti $W_0$ pagal $(W_{-2},W_{-1},W_1,W_2)$.
 - **Nuolatinis skip-gram** veikia priešingai nei CBoW. Modelis naudoja aplinkinių kontekstinių žodžių langelį, kad numatytų esamą žodį.

CBoW yra greitesnis, o skip-gram lėtesnis, bet geriau atstovauja rečiau pasitaikančius žodžius.

![Paveikslėlis rodantis tiek CBoW, tiek Skip-Gram algoritmus, transformuojančius žodžius į vektorius.](../../../../../translated_images/lt/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Norint eksperimentuoti su word2vec įterpimu, iš anksto apmokytu Google News duomenų rinkinyje, galime naudoti **gensim** biblioteką. Žemiau randame žodžius, labiausiai panašius į 'neural'

> **Pastaba:** Pirmą kartą kuriant žodžių vektorius, jų atsisiuntimas gali užtrukti!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Taip pat galime apskaičiuoti žodžių vektorių įgūdžius, kurie bus naudojami klasifikavimo modelio mokymui (aiškumo dėlei rodome tik pirmus 20 vektoriaus komponentų):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Puiku dalykas apie semantines įterptis yra tas, kad galite manipuliuoti vektorinį kodavimą, kad pakeistumėte semantiką. Pavyzdžiui, galime paprašyti rasti žodį, kurio vektorinė reprezentacija būtų kuo artimesnė žodžiams *king* ir *woman*, o kuo toliau nuo žodžio *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Tiek CBoW, tiek Skip-Grams yra „prognozuojančios“ įterptinės, nes jos atsižvelgia tik į vietinius kontekstus. Word2Vec neina naudoti globalaus konteksto.

**FastText** tobulina Word2Vec, mokydamasis vektorių atvaizdavimų kiekvienam žodžiui ir kiekviename žodyje rastiems simbolių n-gramoms. Reprezentacijų reikšmės kiekviename mokymo žingsnyje yra sujungiamos į vieną vektorių. Nors tai prideda daug papildomų skaičiavimų priešmokymui, tai leidžia žodžių įterptinėms koduoti žodžio vidinę informaciją.

Kita metodika, **GloVe**, pasirenka ko-pasirodymo matricos idėją ir naudoja neuroninius metodus, kad sukrautų ko-pasirodymo matricą į išraiškingesnius ir nelinijinius žodžių vektorius.

Galite žaisti su pavyzdžiu, keisdami įterptines į FastText ir GloVe, nes gensim palaiko keletą skirtingų žodžių įterptinių modelių.


## Iš anksto apmokytų įterpimų naudojimas PyTorch

Galime modifikuoti aukščiau pateiktą pavyzdį, kad iš anksto užpildytume mūsų įterpimų sluoksnio matricą semantiniais įterpimais, tokiais kaip Word2Vec. Turime atsižvelgti į tai, kad iš anksto apmokytų įterpimų žodynai ir mūsų teksto korpusas tikriausiai nesutaps, todėl praleidžiamų žodžių svorius inicializuosime atsitiktinėmis reikšmėmis:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Dabar apmokykime mūsų modelį. Atkreipkite dėmesį, kad modelio apmokymo laikas yra žymiai ilgesnis nei ankstesniame pavyzdyje, dėl didesnio įterpimo sluoksnio dydžio ir todėl daug didesnio parametrų skaičiaus. Taip pat dėl šios priežasties gali reikėti apmokyti mūsų modelį su daugiau pavyzdžių, jei norime išvengti per daug pritaikymo prie mokymo duomenų.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Mūsų atveju nematome didelio tikslumo padidėjimo, greičiausiai dėl visiškai skirtingų žodynų. 
Norint įveikti skirtingų žodynų problemą, galime naudoti vieną iš šių sprendimų:
* Iš naujo apmokyti word2vec modelį su mūsų žodynu
* Įkelti mūsų duomenų rinkinį su žodynu iš iš anksto apmokyto word2vec modelio. Žodynas, naudojamas duomenų rinkinio įkėlimui, gali būti nurodytas įkėlimo metu.

Pastarasis metodas atrodo lengvesnis, ypač todėl, kad PyTorch `torchtext` karkasas turi integruotą palaikymą įterpimams. Pavyzdžiui, galime sukurti GloVe pagrindu veikiantį žodyną tokiu būdu:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Užkrautas žodynas turi šias pagrindines operacijas:
* `vocab.stoi` žodynas leidžia mums konvertuoti žodį į jo žodyno indeksą
* `vocab.itos` atlieka priešingą funkciją – konvertuoja skaičių į žodį
* `vocab.vectors` yra įdėjimo vektorių masyvas, tad norint gauti žodžio įdėjimą `s` turime naudoti `vocab.vectors[vocab.stoi[s]]`

Štai pavyzdys, kaip manipuliuoti įdėjimais, kad būtų parodytas lygties **kind-man+woman = queen** veikimas (reikėjo šiek tiek pakoreguoti koeficientą, kad veiktų):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Nor apmokyti klasifikatorių naudojant tuos įterpimus, pirmiausia turime užkoduoti mūsų duomenų rinkinį naudodami GloVe žodyną:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Kaip matėme aukščiau, visi vektoriniai įrašai saugomi `vocab.vectors` matricoje. Tai labai palengvina tų svorių įkėlimą į įterpimo sluoksnio svorius naudojant paprastą kopijavimą:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Dabar išmokykime mūsų modelį ir pažiūrėkime, ar gausime geresnių rezultatų:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Viena iš priežasčių, kodėl nematome reikšmingo tikslumo padidėjimo, yra ta, kad kai kurie mūsų duomenų rinkinio žodžiai trūksta iš anksto apmokytame GloVe žodyne, todėl jie praktiškai yra ignoruojami. Norint šią problemą įveikti, galime apmokyti savo įterpimus ant mūsų duomenų rinkinio. 


## Kontekstinės įterpos

Viena pagrindinių tradicinių iš anksto apmokytų įterpimų reprezentacijų, tokių kaip Word2Vec, apribojimų yra žodžio prasmės atskyrimo problema. Nors iš anksto apmokytos įterpos gali užfiksuoti dalį žodžių reikšmės kontekste, kiekviena galima žodžio reikšmė yra užkoduota toje pačioje įterpoje. Tai gali sukelti problemų tolimesniuose modeliuose, nes daugelio žodžių, pavyzdžiui, žodžio „play“, reikšmės skiriasi priklausomai nuo konteksto, kuriame jie vartojami.

Pavyzdžiui, žodis „play“ šiuose dviejuose sakiniuose turi ganėtinai skirtingas reikšmes:
- Aš nuėjau į **vaidinimą** teatre.
- Jonas nori **žaisti** su savo draugais.

Iš anksto apmokytos įterpos aukščiau atspindi abi šias žodžio „play“ reikšmes toje pačioje įterpoje. Norint įveikti šį apribojimą, turime kurti įterpas, pagrįstas **kalbos modeliu**, kuris yra apmokytas didelės apimties teksto korpuse ir *žino*, kaip žodžiai gali būti sudedami kartu skirtinguose kontekstuose. Diskutuoti apie kontekstines įterpas yra už šio vadovo ribų, tačiau sugrįšime prie jų kalbėdami apie kalbos modelius kitame skyriuje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
